# ***TLC Trip Record Data***

## Installamos las dependencias para poder descargar los datos usando wget

In [3]:
!pip install wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9686 sha256=57cba9bb20b94480e48b0ab5729584445c721d344c90e915c20788ea0e537e5c
  Stored in directory: /home/omar/.cache/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [6]:
pip install pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.5 MB/s  0:00:04m0:00:0100:01
Note: you may need to restart the kernel to use updated packages.


## En este ejemplo solo voy a usar los datos de un solo mes en enero del 2026 y especificamente de los taxis amarillos

In [41]:
import wget
url = 'https://d37ci6vzurychx.cloudfront.net/trip-dTata/yellow_tripdata_2026-01.parquet'
file_output = wget.download(url) 

100% [........................................................................] 64165080 / 64165080

In [1]:
#file_output= 'yellow_tripdata_2026-01.parquet'

## Empezamos a leer la informacion en formato parquet usando pyarrow para poder transformar los datos a una tabla que podemos modificar

In [2]:
import pyarrow.parquet as pq

yellowTripData = pq.read_table(file_output)

pyarrow.Table
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double
----
VendorID: [[2,1,1,2,2,...,2,2,2,2,2],[2,2,2,1,2,...,2,2,2,2,2],...,[2,1,1,1,2,...,2,2,1,2,2],[1,1,1,2,2,...,2,2,2,2,2]]
tpep_pickup_datetime: [[2026-01-01 00:54:04.000000,2026-01-01 00:34:04.000000,2026-01-01 00:57:06.000000,2026-01-01 00:15:22.000000,2026-01-01 00:27:13.000000,...,2026-01-02 20:51:51.000000,2026-01-02 20:34:09.000000,2026-01-02 20:34:09.000000,2026-01-02 20:33:32.000000,2026-01-02 20:44:16.000000],[2026-01-02 20:01:41.000000,2026-01-02 20:26:01.000000,2026-01-02 20:54:53.000000,2026-01-02 20:55:2

# **Hadoop**
## Voy a usar hadoop para almacenar todos los datos que descarguemos y tambien los datos procesados, limpios pero primero hay que configurar la conexion al servidor local de hadoop

In [3]:
import os
import subprocess

os.environ["HADOOP_HOME"] = "/home/omar/opt/hadoop"

os.environ["JAVA_HOME"] = subprocess.check_output(
    "dirname $(dirname $(readlink -f $(which java)))",
    shell=True,
    text=True
).strip()

os.environ["CLASSPATH"] = subprocess.check_output(
    ["hadoop", "classpath"],
    text=True
).strip()

print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print("JAVA_HOME   =", os.environ["JAVA_HOME"])
print("CLASSPATH   =", os.environ["CLASSPATH"][:200], "...")

HADOOP_HOME = /home/omar/opt/hadoop
JAVA_HOME   = /usr/lib/jvm/java-17-openjdk-amd64
CLASSPATH   = /home/omar/opt/hadoop/etc/hadoop:/home/omar/opt/hadoop/share/hadoop/common/lib/*:/home/omar/opt/hadoop/share/hadoop/common/*:/home/omar/opt/hadoop/share/hadoop/hdfs:/home/omar/opt/hadoop/share/hadoop/ ...


## Se crea el directorio donde se va a almacenar la informacion para que despues pyspark pueda leerla desde el servidor de hadoop

In [9]:
import pyarrow.fs as fs

local = fs.LocalFileSystem()

hdfs = fs.HadoopFileSystem(
    host="localhost",
    port=9000
)

In [10]:
hdfs.create_dir("/tlc/raw/yellow", recursive=True)

In [11]:
hdfs_file = "/tlc/raw/yellow/yellow_tripdata_2026-01.parquet"

with open(file_output, "rb") as src:
        with hdfs.open_output_stream(hdfs_file) as dst:
            while chunk := src.read(1024 * 1024):
                dst.write(chunk)


In [12]:
info = hdfs.get_file_info(
    "/tlc/raw/yellow/yellow_tripdata_2026-01.parquet"
)

print(info)

<FileInfo for '/tlc/raw/yellow/yellow_tripdata_2026-01.parquet': type=FileType.File, size=64165080>


# **Spark**
## Creamos la session de spark con la que vamos a realizar el analisis de los datos y leemos los datos desde el servidor de Hadoop donde vamos a tener toda nuestra informacion almacenada

In [49]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NYCTaxiAnalysis")
    .config(
        "spark.jars.packages",
        "com.mysql:mysql-connector-j:9.4.0"
    )
    .getOrCreate()
)

df = spark.read.parquet(
    "hdfs://localhost:9000/tlc/raw/yellow/yellow_tripdata_2026-01.parquet"
)

df.printSchema()
#df.show(10)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## Contamos la cantidad de datos que tenemos

In [48]:
print("Number of rows:", df.count())

df.describe()

Number of rows: 3724889


DataFrame[summary: string, VendorID: string, passenger_count: string, trip_distance: string, RatecodeID: string, store_and_fwd_flag: string, PULocationID: string, DOLocationID: string, payment_type: string, fare_amount: string, extra: string, mta_tax: string, tip_amount: string, tolls_amount: string, improvement_surcharge: string, total_amount: string, congestion_surcharge: string, Airport_fee: string, cbd_congestion_fee: string]

## Contamos los valores null dentro de las columnas

In [53]:
from pyspark.sql.functions import col, sum
null_counts = df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]
)
null_counts.select("passenger_count","RatecodeID","store_and_fwd_flag","Airport_fee","congestion_surcharge").show()

+---------------+----------+------------------+-----------+--------------------+
|passenger_count|RatecodeID|store_and_fwd_flag|Airport_fee|congestion_surcharge|
+---------------+----------+------------------+-----------+--------------------+
|        1088058|   1088058|           1088058|    1088058|             1088058|
+---------------+----------+------------------+-----------+--------------------+



## Podemos ver que tenemos al menos un millon de datos con valores nulos en algunas columnas, la mas importante seria passenger_count ya que nos indica la cantidad de pasajeros que habia durante el viaje

## Obtenemos informacion importante de los la distancia, cuanto pagaron en total y el numero de pasajeros

In [16]:
df.select(
    "trip_distance",
    "fare_amount",
    "payment_type",
    "total_amount",
    "passenger_count"
).describe().show()

+-------+-----------------+------------------+------------------+------------------+------------------+
|summary|    trip_distance|       fare_amount|      payment_type|      total_amount|   passenger_count|
+-------+-----------------+------------------+------------------+------------------+------------------+
|  count|          3724889|           3724889|           3724889|           3724889|           2636831|
|   mean|6.455646860885151|20.804253893199448|0.8465637499533543|29.178525515798285| 1.256271258946819|
| stddev|648.8855284529166|18.927007021274658|0.7120492979865746|22.585529763602636|0.6702431378098668|
|    min|              0.0|           -2555.2|                 0|           -2560.2|                 0|
|    max|        269097.48|            2555.2|                 4|            2560.2|                 9|
+-------+-----------------+------------------+------------------+------------------+------------------+



In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    hour,
    avg,
    sum,
    count,
    unix_timestamp,
    date_format,
    dayofwee#,
    avg,
    sum
)

## Limpiamos un poco los datos por ejemplo eliminamos los datos negativos o menores a cero en la distancia, el numero de pasajeros y el pago total

In [54]:
clean_df = (
    df
    .filter(col("trip_distance") > 0)
    .filter(col("total_amount") > 0)
    .filter(col("passenger_count") > 0)
)

## Duracion promedio de los viajes en minutos

In [55]:
time_min_df = df.withColumn(
    "trip_duration_minutes",
    (
        unix_timestamp("tpep_dropoff_datetime")
        - unix_timestamp("tpep_pickup_datetime")
    ) / 60
)
time_min_df = time_min_df.filter(
    (col("trip_duration_minutes") > 0) &
    (col("trip_duration_minutes") < 600)
)
time_min_df.select("trip_duration_minutes").describe().show()

+-------+---------------------+
|summary|trip_duration_minutes|
+-------+---------------------+
|  count|              3678797|
|   mean|   17.088416856560897|
| stddev|   14.572335015985303|
|    min| 0.016666666666666666|
|    max|                599.6|
+-------+---------------------+



## Informacion acerca general acerca de la hora en la toman los taxis, el dia de la semana y la fecha

In [56]:
pickup_info = (
    clean_df
    .withColumn(
        "pickup_hour",
        hour("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_day_of_week",
        dayofweek("tpep_pickup_datetime")
    )
    .withColumn(
        "pickup_date",
        date_format(
            "tpep_pickup_datetime",
            "yyyy-MM-dd"
        )
    )
)

pickup_info.select("pickup_hour","pickup_day_of_week","pickup_date").describe().show()

[Stage 99:=================================================>      (14 + 2) / 16]

+-------+------------------+------------------+-----------+
|summary|       pickup_hour|pickup_day_of_week|pickup_date|
+-------+------------------+------------------+-----------+
|  count|           2552307|           2552307|    2552307|
|   mean|14.359021857480311|  4.42049330272573|       NULL|
| stddev| 5.524580320058522|1.8977893758720095|       NULL|
|    min|                 0|                 1| 2025-12-31|
|    max|                23|                 7| 2026-02-01|
+-------+------------------+------------------+-----------+



## El numero de viajes por hora

In [21]:
trips_by_hour = (
    clean_df
    .groupBy("pickup_hour")
    .count()
    .orderBy("pickup_hour")
)

trips_by_hour.show()

+-----------+------+
|pickup_hour| count|
+-----------+------+
|          0|110883|
|          1| 77084|
|          2| 53616|
|          3| 38334|
|          4| 28480|
|          5| 32710|
|          6| 63039|
|          7|111974|
|          8|146154|
|          9|154551|
|         10|159026|
|         11|169275|
|         12|184866|
|         13|192861|
|         14|204508|
|         15|218593|
|         16|224918|
|         17|249006|
|         18|262252|
|         19|231761|
+-----------+------+
only showing top 20 rows


## El precio por hora

In [22]:
fare_by_hour = (
    clean_df
    .groupBy("pickup_hour")
    .agg(
        avg("fare_amount").alias("average_fare"),
        avg("trip_distance").alias("average_distance")
    )
    .orderBy("pickup_hour")
)

fare_by_hour.show()

+-----------+------------------+------------------+
|pickup_hour|      average_fare|  average_distance|
+-----------+------------------+------------------+
|          0|21.536124653913646| 4.758033693172083|
|          1|21.398978776399385|  8.29400978153704|
|          2|20.241790696807293| 9.225093255744543|
|          3|20.752587259352214| 9.967425262169346|
|          4|23.846395716292303| 7.524356039325847|
|          5|26.348777132375464| 15.28087343320087|
|          6|25.431057916528168| 9.330351052523028|
|          7|23.741841677531433|17.544190615678634|
|          8|22.445412920616867| 8.775000273683915|
|          9|20.427450744413623| 7.499634942510854|
|         10|19.663883453019597|  9.22966653251669|
|         11| 19.37801589130051| 7.073521902230104|
|         12|19.699715685955823| 5.338267501866235|
|         13| 19.99243216617085|  5.57596590290418|
|         14|20.886962954993702|6.6418544506816195|
|         15|21.153232445685816| 5.295731519307579|
|         16

## Las ganancias por dia

In [23]:
revenue_by_day = (
    clean_df
    .groupBy("pickup_date")
    .agg(
        sum("total_amount").alias("total_revenue"),
        count("*").alias("total_trips")
    )
    .orderBy("pickup_date")
)

print("Number of rows:", revenue_by_day.count())
revenue_by_day.show()

Number of rows: 33
+-----------+------------------+-----------+
|pickup_date|     total_revenue|total_trips|
+-----------+------------------+-----------+
| 2025-12-31|            258.35|          6|
| 2026-01-01| 3577221.169999766|     113213|
| 2026-01-02|2924616.1299998797|      98967|
| 2026-01-03| 3069915.229999866|     107232|
| 2026-01-04| 2775485.689999897|      92403|
| 2026-01-05|2882279.3699999233|      95998|
| 2026-01-06|3041623.4399999976|     106295|
| 2026-01-07| 3171778.349999929|     111717|
| 2026-01-08|3402970.2199998936|     118001|
| 2026-01-09| 3479686.459999881|     122493|
| 2026-01-10|3890800.0799997393|     143530|
| 2026-01-11|3213098.2799998205|     114497|
| 2026-01-12| 3198257.229999924|     111244|
| 2026-01-13|3497242.2699999083|     121667|
| 2026-01-14|3681227.2499999115|     128328|
| 2026-01-15|3981673.7299998836|     139510|
| 2026-01-16|  3788093.25999986|     132960|
| 2026-01-17| 3332677.599999902|     126593|
| 2026-01-18| 3178227.659999796|    

## Los metodos de pagos mas utilizados

In [24]:
payment_analysis = (
    clean_df
    .groupBy("payment_type")
    .agg(
        count("*").alias("total_trips"),
        avg("fare_amount").alias("average_fare"),
        avg("tip_amount").alias("average_tip")
    )
    .orderBy("payment_type")
)

payment_analysis.show()

+------------+-----------+------------------+--------------------+
|payment_type|total_trips|      average_fare|         average_tip|
+------------+-----------+------------------+--------------------+
|           0|    1087995| 25.37964413438449|  0.3750508871823849|
|           1|    2209718|19.614287759796255|   4.139539085982484|
|           2|     308707|18.168181900637354|4.252899999028204...|
|           3|      16188|6.2723023227081764|                 0.0|
|           4|      56189|1.1683571517556826|1.797504849703678...|
+------------+-----------+------------------+--------------------+



## El lugar mas comun donde toman el taxi

In [25]:
pickup_analysis = (
    clean_df
    .groupBy("PULocationID")
    .count()
    .orderBy("count", ascending=False)
)
pickup_analysis.show()

+------------+------+
|PULocationID| count|
+------------+------+
|         237|157000|
|         132|151289|
|         236|150131|
|         161|144185|
|         186|108884|
|         142|107880|
|         162|106970|
|         230|104772|
|          79|100222|
|         239| 95664|
|         234| 93939|
|         170| 91871|
|          68| 89768|
|         141| 85803|
|         163| 84606|
|         138| 84117|
|          48| 81922|
|         249| 78562|
|         140| 76206|
|         263| 72858|
+------------+------+
only showing top 20 rows


## La columna con el ID no nos dice mucho por si solo asi que tenemos que incluir otro archivo donde tenemos mas informacion acerca de estos id

In [26]:
from pyarrow import csv

taxi_zone_file = "taxi_zone_lookup.csv"
hdfs_file = "/tlc/raw/yellow/taxi_zone_lookup.csv"

with open(taxi_zone_file, "rb") as src:
        with hdfs.open_output_stream(hdfs_file) as dst:
            while chunk := src.read(1024 * 1024):
                dst.write(chunk)


## Despues de subir este archivo a hadoop lo leemos desde pyspark

In [27]:
df_zn = spark.read.csv("hdfs://localhost:9000/tlc/raw/yellow/taxi_zone_lookup.csv")
df_zn= df_zn.withColumnRenamed("_c0","LocationID").withColumnRenamed("_c1","Borough").withColumnRenamed("_c2","Zone").withColumnRenamed("_c3","Service_zone")
df_zn =df_zn.filter(df_zn.LocationID !="LocationID")


## Con esto podemos analizar el numero de viajes por zona usando el LocationID

In [28]:
pickup_zones = (
    clean_df
    .groupBy("PULocationID")
    .agg(count("*").alias("total_pickups"))
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .select(
        "LocationID",
        "Borough",
        "Zone",
        "total_pickups"
    )
    .orderBy("total_pickups", ascending=False)
)

pickup_zones.show(20, truncate=False)

+----------+---------+----------------------------+-------------+
|LocationID|Borough  |Zone                        |total_pickups|
+----------+---------+----------------------------+-------------+
|237       |Manhattan|Upper East Side South       |157000       |
|132       |Queens   |JFK Airport                 |151289       |
|236       |Manhattan|Upper East Side North       |150131       |
|161       |Manhattan|Midtown Center              |144185       |
|186       |Manhattan|Penn Station/Madison Sq West|108884       |
|142       |Manhattan|Lincoln Square East         |107880       |
|162       |Manhattan|Midtown East                |106970       |
|230       |Manhattan|Times Sq/Theatre District   |104772       |
|79        |Manhattan|East Village                |100222       |
|239       |Manhattan|Upper West Side South       |95664        |
|234       |Manhattan|Union Sq                    |93939        |
|170       |Manhattan|Murray Hill                 |91871        |
|68       

## Destinos con la mayor cantidad de viajes

In [29]:
dropoff_zones = (
    clean_df
    .groupBy("DOLocationID")
    .agg(count("*").alias("total_dropoffs"))
    .join(
        df_zn,
        clean_df.DOLocationID == df_zn.LocationID
    )
    .select(
        "LocationID",
        "Borough",
        "Zone",
        "total_dropoffs"
    )
    .orderBy("total_dropoffs", ascending=False)
)

dropoff_zones.show(20, truncate=False)

+----------+---------+-----------------------------+--------------+
|LocationID|Borough  |Zone                         |total_dropoffs|
+----------+---------+-----------------------------+--------------+
|236       |Manhattan|Upper East Side North        |153723        |
|237       |Manhattan|Upper East Side South        |144064        |
|161       |Manhattan|Midtown Center               |119674        |
|170       |Manhattan|Murray Hill                  |98317         |
|230       |Manhattan|Times Sq/Theatre District    |97791         |
|142       |Manhattan|Lincoln Square East          |96024         |
|239       |Manhattan|Upper West Side South        |95700         |
|141       |Manhattan|Lenox Hill West              |94387         |
|68        |Manhattan|East Chelsea                 |89373         |
|79        |Manhattan|East Village                 |88272         |
|162       |Manhattan|Midtown East                 |88100         |
|234       |Manhattan|Union Sq                  

## Numero de viajes por Borough

In [30]:
pickup_boroughs = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy("Borough")
    .agg(
        count("*").alias("total_pickups")
    )
    .orderBy("total_pickups", ascending=False)
)

pickup_boroughs.show()

+-------------+-------------+
|      Borough|total_pickups|
+-------------+-------------+
|    Manhattan|      3137719|
|       Queens|       342073|
|     Brooklyn|       155373|
|        Bronx|        36782|
|      Unknown|         4372|
|          N/A|         1509|
|Staten Island|          485|
|          EWR|          484|
+-------------+-------------+



## Este analisis para ver los viajes mas comunes de cierta zona a otra

In [31]:
pickup_zones_df = df_zn.alias("pickup")
dropoff_zones_df = df_zn.alias("dropoff")

routes = (
    clean_df
    .join(
        df_zn,
        (clean_df.DOLocationID == df_zn.LocationID) #& (clean_df.DOLocationID == df_zn.LocationID)
    )
    .groupBy(
        clean_df.PULocationID.alias("pickup_zone"),
        clean_df.DOLocationID.alias("dropoff_zone"),
        df_zn.Zone
    )
    .agg(
        count("*").alias("total_trips")
    )
    .orderBy("total_trips", ascending=False)
)

routes.show(20, truncate=False)

[Stage 59:==========================================>             (12 + 4) / 16]

+-----------+------------+-------------------------+-----------+
|pickup_zone|dropoff_zone|Zone                     |total_trips|
+-----------+------------+-------------------------+-----------+
|237        |236         |Upper East Side North    |23619      |
|236        |237         |Upper East Side South    |20536      |
|236        |236         |Upper East Side North    |16882      |
|237        |237         |Upper East Side South    |15782      |
|161        |237         |Upper East Side South    |10189      |
|237        |161         |Midtown Center           |9342       |
|142        |239         |Upper West Side South    |8870       |
|239        |238         |Upper West Side North    |8695       |
|161        |236         |Upper East Side North    |8410       |
|239        |142         |Lincoln Square East      |8385       |
|141        |236         |Upper East Side North    |8368       |
|236        |141         |Lenox Hill West          |8033       |
|237        |141         

## Precios promedios por zona

In [32]:
fare_by_zone = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone"
    )
    .agg(
        count("*").alias("total_trips"),
        avg("fare_amount").alias("average_fare"),
        avg("total_amount").alias("average_total")
    )
    .orderBy("average_total", ascending=False)
)

fare_by_zone = fare_by_zone.filter(
    (col("total_trips") > 10)
)
fare_by_zone.show(20, truncate=False)

+-------------+-----------------------------------+-----------+------------------+------------------+
|Borough      |Zone                               |total_trips|average_fare      |average_total     |
+-------------+-----------------------------------+-----------+------------------+------------------+
|EWR          |Newark Airport                     |484        |89.54012396694216 |103.84373966942152|
|N/A          |Outside of NYC                     |1509       |87.61176938369779 |99.31344599072229 |
|Staten Island|Great Kills                        |15         |66.69533333333332 |79.85466666666667 |
|Staten Island|West Brighton                      |13         |61.05692307692307 |74.60000000000001 |
|Queens       |JFK Airport                        |151289     |55.97117913397541 |72.41148146923324 |
|Queens       |Flushing Meadows-Corona Park       |439        |57.24243735763098 |70.0974487471526  |
|Queens       |LaGuardia Airport                  |84117      |42.191832685426306|

## Distancia promedio por zona

In [33]:
distance_by_zone = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone"
    )
    .agg(
        count("*").alias("total_trips"),
        avg("trip_distance").alias("average_distance")
    )
    .orderBy("average_distance", ascending=False)
)

distance_by_zone.show(20, truncate=False)

+---------+----------------------------+-----------+------------------+
|Borough  |Zone                        |total_trips|average_distance  |
+---------+----------------------------+-----------+------------------+
|Brooklyn |Columbia Street             |444        |193.2281981981983 |
|Queens   |Auburndale                  |198        |170.76409090909092|
|Bronx    |Hunts Point                 |794        |140.65827455919387|
|Queens   |Elmhurst/Maspeth            |1147       |105.95526591107239|
|Manhattan|Washington Heights North    |4482       |99.24108210620273 |
|Queens   |Saint Albans                |1456       |75.28640796703289 |
|Brooklyn |Brooklyn Heights            |5058       |68.28932779754845 |
|Queens   |Jackson Heights             |4932       |61.53401865369015 |
|Brooklyn |East Flatbush/Farragut      |2263       |57.478281042863365|
|Queens   |Steinway                    |3402       |47.81860082304529 |
|Bronx    |Soundview/Bruckner          |867        |47.123967704

## Ganancias totales por zona

In [34]:
revenue_by_zone = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone"
    )
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
)

revenue_by_zone.show(20, truncate=False)

+---------+----------------------------+-----------+--------------------+
|Borough  |Zone                        |total_trips|total_revenue       |
+---------+----------------------------+-----------+--------------------+
|Queens   |JFK Airport                 |151289     |1.0955060619998828E7|
|Queens   |LaGuardia Airport           |84117      |5579972.889999996   |
|Manhattan|Midtown Center              |144185     |3798181.439999952   |
|Manhattan|Upper East Side North       |150131     |3367330.629999923   |
|Manhattan|Upper East Side South       |157000     |3363570.749999945   |
|Manhattan|Times Sq/Theatre District   |104772     |3030427.559999987   |
|Manhattan|Penn Station/Madison Sq West|108884     |2745606.0199999693  |
|Manhattan|Midtown East                |106970     |2716745.3399999905  |
|Manhattan|East Village                |100222     |2502446.409999831   |
|Manhattan|Lincoln Square East         |107880     |2492377.519999976   |
|Manhattan|East Chelsea               

## Numero de viajes por zona y por hora del dia

In [35]:
zone_hour = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Zone",
        "Borough",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips")
    )
    .orderBy(
        "Zone",
        "pickup_hour",
         ascending=False
    )
)
zone_hour.show(20, truncate=False)

+--------------+---------+-----------+-----------+
|Zone          |Borough  |pickup_hour|total_trips|
+--------------+---------+-----------+-----------+
|Yorkville West|Manhattan|23         |2194       |
|Yorkville West|Manhattan|22         |3297       |
|Yorkville West|Manhattan|21         |3655       |
|Yorkville West|Manhattan|20         |3920       |
|Yorkville West|Manhattan|19         |4575       |
|Yorkville West|Manhattan|18         |5302       |
|Yorkville West|Manhattan|17         |4687       |
|Yorkville West|Manhattan|16         |4220       |
|Yorkville West|Manhattan|15         |4145       |
|Yorkville West|Manhattan|14         |4054       |
|Yorkville West|Manhattan|13         |3993       |
|Yorkville West|Manhattan|12         |3863       |
|Yorkville West|Manhattan|11         |3790       |
|Yorkville West|Manhattan|10         |3871       |
|Yorkville West|Manhattan|9          |3608       |
|Yorkville West|Manhattan|8          |3429       |
|Yorkville West|Manhattan|7    

## Ganancias promedio por zona y por hora

In [36]:
zone_hour_revenue = (
    clean_df
    .join(
        df_zn,
        clean_df.PULocationID == df_zn.LocationID
    )
    .groupBy(
        "Borough",
        "Zone",
        "pickup_hour"
    )
    .agg(
        count("*").alias("total_trips"),
        sum("total_amount").alias("total_revenue"),
        avg("total_amount").alias("average_fare")
    )
    .orderBy(
        "total_revenue",
        "average_fare",
         ascending=False
    )
)
zone_hour_revenue.show(20, truncate=False)

+-------+-----------------+-----------+-----------+------------------+-----------------+
|Borough|Zone             |pickup_hour|total_trips|total_revenue     |average_fare     |
+-------+-----------------+-----------+-----------+------------------+-----------------+
|Queens |JFK Airport      |16         |11260      |934572.040000015  |82.9992930728255 |
|Queens |JFK Airport      |20         |11932      |854773.2700000078 |71.63704911163325|
|Queens |JFK Airport      |15         |10349      |815600.6300000073 |78.8096076915651 |
|Queens |JFK Airport      |19         |10584      |809142.9600000025 |76.44963718820885|
|Queens |JFK Airport      |17         |9062       |740182.0199999994 |81.67976384903989|
|Queens |JFK Airport      |21         |10592      |714815.030000001  |67.4863132552871 |
|Queens |JFK Airport      |18         |8981       |709274.9200000018 |78.9750495490482 |
|Queens |JFK Airport      |22         |10311      |708095.390000003  |68.67378430802086|
|Queens |JFK Airport 

# **MySQL**
## Por ultimo guardamos toda esta informacion dentro de nuestra base de datos para despues mostrarla desde streamlit

In [37]:
jdbc_url = "jdbc:mysql://localhost:3306/taxi_db"

properties = {
    "user": "spark",
    "password": "spark",
    "driver": "com.mysql.cj.jdbc.Driver"
}

In [38]:
spark.conf.get("spark.jars.packages")

'com.mysql:mysql-connector-j:9.4.0'

In [39]:
test_df = spark.read.jdbc(
    url=jdbc_url,
    table="test_table",
    properties=properties
)

test_df.show()

+---+
| ID|
+---+
+---+



In [40]:
#clean_df.write.jdbc(
   # url=jdbc_url,
   # table="taxi_trips",
   # mode="overwrite",
   # properties=properties
#)